# 007 — Basic RAG Pipeline (End-to-End)
**Retrieval Series** | Load → Split → Embed → Store → Retrieve → Generate

The complete RAG pipeline in a single notebook.
Every advanced technique builds on top of this foundation.


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile
✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)


In [3]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


✓ 5 dataset files loaded
  File                    Chars   ~Words
  ----------------------------------------
  ai.txt                  6,221      943
  machine_learning.txt    5,953      870
  nlp.txt                 5,160      693
  deep_learning.txt       5,114      686
  rag.txt                 5,146      715
  ----------------------------------------
  TOTAL                  27,602    3,907


In [4]:
# ── STEP 1: Load & Split ─────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Use all 5 downloaded articles
raw_docs = {
    "ai":  ai_text,
    "ml":  ml_text,
    "nlp": nlp_text,
    "dl":  dl_text,
    "rag": rag_text,
}

all_docs = []
for source, text in raw_docs.items():
    for chunk in splitter.split_text(text[:20000]):
        all_docs.append(Document(page_content=chunk, metadata={"source": source}))

print(f"✓ {len(all_docs)} chunks from {len(raw_docs)} sources")


✓ 75 chunks from 5 sources


In [5]:
# ── STEP 2: Embed & Store ────────────────────────────────────────────────
import time
from langchain_community.vectorstores import FAISS

t0 = time.time()
vectorstore = FAISS.from_documents(all_docs, embeddings)
print(f"✓ Vectorstore built in {time.time()-t0:.1f}s with {len(all_docs)} vectors")

# Save to disk for reuse
import os
vs_path = os.path.join(os.getcwd(), '..', 'datasets', 'basic_rag_index')
vectorstore.save_local(vs_path)
print(f"✓ Saved to {vs_path}")


✓ Vectorstore built in 1.3s with 75 vectors
✓ Saved to /home/saurabh/projects/personal_project/advance-rag/notebooks/../datasets/basic_rag_index


In [6]:
# ── STEP 3: Build Retriever ──────────────────────────────────────────────
retriever = vectorstore.as_retriever(
    search_type="mmr",          # Maximum Marginal Relevance → diverse results
    search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.7}
)
print("✓ MMR retriever ready  (k=5, diversity λ=0.7)")

# Quick test
test_results = retriever.invoke("What is reinforcement learning?")
print(f"\nRetrieved {len(test_results)} docs:")
for r in test_results:
    print(f"  [{r.metadata['source']}] {r.page_content[:100]}...")


✓ MMR retriever ready  (k=5, diversity λ=0.7)

Retrieved 5 docs:
  [ai] Reinforcement learning (RL) is an area of machine learning concerned with how intelligent
agents oug...
  [ml] Reinforcement Learning

Reinforcement learning (RL) is about learning to make decisions by interacti...
  [ml] Q-learning is a model-free RL algorithm that learns the value of actions in states. Deep Q-Network
(...
  [ai] Machine Learning Approaches

Machine learning is a subfield of AI. It is a method of data analysis t...
  [ml] The Markov Decision Process (MDP) is the mathematical framework for RL. It consists of:
- States (S)...


In [7]:
# ── STEP 4: RAG Chain ────────────────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[Source: {d.metadata['source']}]\n{d.page_content}" for d in docs
    )

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a helpful assistant. Answer the question using ONLY the provided context.\n"
        "If the context does not contain enough information, say so.\n"
        "Always cite the source in brackets, e.g. [ai], [ml].\n\n"
        "Context:\n{context}"
    )),
    ("human", "{question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)
print("✓ RAG chain assembled")


✓ RAG chain assembled


In [8]:
# ── STEP 5: Ask Questions ─────────────────────────────────────────────────
questions = [
    "What is the difference between supervised and unsupervised learning?",
    "How does attention mechanism work in transformers?",
    "What is RAG and why is it used with LLMs?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print("-" * 60)


Q: What is the difference between supervised and unsupervised learning?


A: Supervised learning is the machine learning task of learning a function that maps an input to an output based on example input-output pairs. It infers a function from labeled training data consisting of a set of training examples [ai].

Unsupervised learning, on the other hand, is a type of algorithm that learns patterns from untagged data. The hope is that through mimicry, which is computationally cheaper than other approaches, the machine is forced to build a compact internal representation of its world [ai].
------------------------------------------------------------
Q: How does attention mechanism work in transformers?


A: The attention mechanism in transformers computes a query-key-value system:

Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V

where:
- Q is the query
- K is the key
- V is the value
- d_k is the dimensionality of the key vector

This allows the model to jointly attend to information from different positions at different representation levels. [dl]
------------------------------------------------------------
Q: What is RAG and why is it used with LLMs?


A: RAG stands for Retrieval-Augmented Generation. It is a hybrid AI architecture that combines information retrieval with text generation. RAG systems retrieve relevant documents from an external knowledge base and use them as context for generating responses.

RAG is used with LLMs (Language Models) to address key limitations of parametric (weights-only) language models: knowledge staleness, hallucination, and lack of attribution. [Source: rag]
------------------------------------------------------------


In [9]:
# ── STEP 6: Source attribution ───────────────────────────────────────────
def rag_with_sources(question: str):
    docs = retriever.invoke(question)
    sources = list({d.metadata["source"] for d in docs})
    answer  = rag_chain.invoke(question)
    return {"answer": answer, "sources": sources, "doc_count": len(docs)}

result = rag_with_sources("What role does backpropagation play in deep learning?")
print(f"Answer  : {result['answer'][:400]}")
print(f"Sources : {result['sources']}")
print(f"Docs used: {result['doc_count']}")


Answer  : Backpropagation is the algorithm used to compute gradients in neural networks. It applies the chain rule of calculus to propagate error signals backwards through the network, computing how much each weight contributed to the overall error. [ml]
Sources : ['dl', 'ml']
Docs used: 5


## Key Takeaways
- **Load → Split → Embed → Store → Retrieve → Generate** is the core loop
- MMR retrieval gives more **diverse** results vs pure similarity
- Always include source attribution — it lets users verify answers
- Next steps: swap FAISS for hybrid (006), add reranking (009), add self-critique (010)
